# How do you translate model results into a business case?
**Topics:** Lift to Business Value · Implementation Cost · Inference Cost · Break-Even Analysis · Opportunity Cost

---
## Abbreviation Reference

| Abbreviation | Full Name |
|---|---|
| AUC | Area Under the Curve |
| CAC | Customer Acquisition Cost |
| CI | Confidence Interval |
| CTR | Click-Through Rate |
| LTV | Lifetime Value |
| MDE | Minimum Detectable Effect |
| ML | Machine Learning |
| MRR | Monthly Recurring Revenue |
| ROI | Return on Investment |
| SLA | Service Level Agreement |

---
## 1. Translating lift into business value

### The core formula
- Business value = measured lift × affected volume × value per unit
- Always apply lift to the correct base — targeted users, not all users

### By outcome type

| Outcome type | Formula | Example |
|---|---|---|
| Revenue impact | lift in conversion rate × targeted users × average order value | +2% conversion × 100k users × $50 AOV = $100k |
| Cost savings | reduction in error rate × volume of decisions × cost per error | −5% false negative rate × 10k cases × $200 cost = $100k |
| Risk reduction | reduction in false negative rate × volume × cost of a miss | −3% miss rate × 50k events × $500 = $75k |
| Engagement | lift in retention × churned users saved × LTV per user | +1% retention × 5k users × $300 LTV = $15k |

### Common mistakes

| Mistake | Why it matters |
|---|---|
| Applying relative lift to wrong base | "10% lift" on 1% baseline = 0.1% absolute — sounds big, is small |
| Using point estimate without Confidence Interval (CI) | Lower bound of CI may be below break-even |
| Ignoring segment heterogeneity | Average lift across all users hides that the model helps some and hurts others |
| Double-counting with existing models | New model replaces old model — the gain is the delta, not the full lift |
| Using training-set lift in the business case | Always use held-out or A/B lift — training lift is optimistic |

### Gotchas
- Always use the lower bound of the 95% CI as the conservative estimate in the business case
- If the lower bound is below break-even, the result is not strong enough to justify investment

---
## 2. Implementation cost

### One-time costs

| Cost item | Questions to ask |
|---|---|
| Engineering time | How many weeks of eng time to integrate, test, and deploy? |
| Infrastructure changes | New serving stack, database schema changes, API changes? |
| Data pipeline changes | New features require new data sources or ETL jobs? |
| Migration | What happens to existing model predictions during cutover? |
| Labeling | Does the new model require labeled data that doesn't yet exist? |

### Ongoing costs

| Cost item | Questions to ask |
|---|---|
| Retraining cadence | Weekly, monthly, or on-trigger? Each run has compute and engineering cost |
| Monitoring | Who owns the dashboard? What is the on-call burden? |
| Maintenance | Feature drift, dependency updates, model decay management |
| Human review loops | Does the model output require human review before action? |

### Gotchas
- Hidden costs (monitoring, maintenance, labeling) often exceed the initial build cost over 12 months
- "Simple" integrations frequently uncover downstream dependencies that multiply eng time

---
## 3. Inference cost

### Latency vs. accuracy tradeoff

| Serving mode | Latency target | Model complexity constraint |
|---|---|---|
| Real-time (user-facing) | < 100ms Service Level Agreement (SLA) | Lightweight models only; deep learning often infeasible |
| Near real-time (async) | < 1s | Moderate complexity acceptable |
| Batch (offline scoring) | Hours | No latency constraint; large models feasible |

### Cost per prediction
- Estimate: (compute cost per hour × inference time per prediction × daily prediction volume × 365)
- Compare: does the annual inference cost fit within the expected Return on Investment (ROI)?
- Larger model ≠ better ROI — accuracy-cost tradeoff must be explicit

### Accuracy vs. cost decision table

| Scenario | Recommendation |
|---|---|
| Small model, small lift | Only ship if implementation cost is also very low |
| Small model, large lift | Ship — best ROI |
| Large model, small lift | Do not ship — inference cost likely exceeds value |
| Large model, large lift | Evaluate carefully — check if a smaller model captures most of the lift |

### Gotchas
- Always benchmark a simpler baseline before committing to a complex model
- A model that is 2% more accurate but 10× more expensive to serve is rarely the right choice

---
## 4. Break-even analysis

### The formula
- Minimum lift needed = total cost / (affected volume × value per unit lift)
- If measured lift (lower CI bound) < minimum lift needed → do not ship

### Worked example

| Input | Value |
|---|---|
| Total cost (one-time + 12-month ongoing) | $200,000 |
| Affected volume (monthly users) | 500,000 |
| Value per conversion | $40 |
| Current conversion rate | 5% |
| Minimum lift needed | $200k / (500k × 0.05 × $40) = 2.0% relative lift |
| Measured lift (lower CI bound) | 1.2% relative |
| Decision | Do not ship — measured lift is below break-even |

### Sensitivity table
- Run break-even across a range of cost and volume assumptions
- Identifies which assumptions the decision is most sensitive to

| If cost is... | Break-even lift needed |
|---|---|
| $100k | 1.0% relative |
| $200k | 2.0% relative |
| $400k | 4.0% relative |

### Gotchas
- Do not use the point estimate of lift in the business case — use the lower bound of the 95% CI
- Break-even is a floor, not a target — a model that barely breaks even often isn't worth the operational burden

---
## 5. Opportunity cost

### What it is
- Every engineering week spent on this model is a week not spent on something else
- Opportunity cost is real even when the model has positive ROI

### Questions to ask

| Question | Why it matters |
|---|---|
| What else could the team build in the same time? | A new feature may deliver more value than a marginal model improvement |
| Is this the highest-leverage ML problem available? | Prioritize problems with large volume × high value per unit |
| Are we in diminishing returns on this model? | If the last three improvements were each smaller, it may be time to move on |
| Does this unblock other work or create dependencies? | Shared infrastructure investments have leverage; one-off models don't |

### The "good enough" threshold
- Define upfront the performance level at which further improvement is not worth the cost
- Once a model crosses this threshold, redirect effort to the next highest-value problem
- Typical signal: marginal gains from model iteration are smaller than noise in production metrics

### Gotchas
- Sunk cost fallacy: "we've already invested 3 months" is not a reason to continue a low-ROI project
- A model that works but nobody acts on delivers zero value — validate the decision workflow before building

---
## 6. Decision table

### Combined signal framework

| Lift confidence | Cost | Break-even met? | Recommendation |
|---|---|---|---|
| High (tight CI, above MDE) | Low | Yes | Ship |
| High | High | Yes | Ship — monitor inference cost closely |
| High | Low | No | Revisit cost assumptions or reduce scope |
| Moderate (wide CI, borderline) | Low | Yes | Ship with canary rollout and guardrails |
| Moderate | High | Yes | Reduce model complexity first, re-evaluate |
| Low (CI includes zero) | Any | No | Do not ship — gather more evidence |

### When to revisit vs. abandon

| Situation | Action |
|---|---|
| Lift is real but below break-even | Reduce cost (simpler model, less infra) or increase volume (broader targeting) |
| Break-even met but opportunity cost is high | Deprioritize — ship only if it unblocks other work |
| Multiple iterations with diminishing gains | Abandon — set "good enough" threshold and move on |
| No A/B validation possible | Raise evidence bar — require stronger causal validation before shipping |